#### 전체 언어 모델링 (FLM)

In [ ]:
import torch
from transformers import AutoTokenizer, GPTNeoForCausalLM

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")
model = GPTNeoForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")

inputs = tokenizer("Language models are", return_tensors="pt")
gen_tokens = model.generate(**inputs, max_new_tokens=1, output_scores=True, return_dict_in_generate=True)
output_scores = gen_tokens["scores"]
scores_tensor = output_scores[0]
sorted_indices = torch.argsort(scores_tensor[0], descending=True)[:20]

for index in sorted_indices:
    token_id = index
    token_name = tokenizer.decode([token_id.item()])
    token_score = scores_tensor[0][index].item()
    print(f"Token: {token_name}, Score: {token_score}")

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.31G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

#### OpenAI API

In [2]:
from openai import OpenAI

In [ ]:
client = OpenAI(api_key=...)

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user",
        "content": "I had 25 eggs. I gave away 12. I now have"}
    ],
    max_tokens=1,
    temperature=0,
    logprobs=True,
    top_logprobs=10
)

print(f"Generated: {response.choices[0].message.content}")
print("Top probabilities:")
for token in response.choices[0].logprobs.content[0].top_logprobs:
    print(f"'{token.token}': {token.logprob:.4f}")

#### 마스크 언어 모델링 (MLM)

In [ ]:
import torch
from transformers import AutoTokenizer, BertForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

text = "Language models are powerful tools for natural language processing."
inputs = tokenizer(text, return_tensors="pt")

# targets: 원본 token ids 복사, 마스킹되지 않은 위치는 -100 (loss 계산 제외)
targets = inputs["input_ids"].clone()
targets[:] = -100

# [MASK] 처리: "powerful" 위치를 [MASK]로 교체
mask_token_index = (inputs["input_ids"] == tokenizer.convert_tokens_to_ids("powerful")).nonzero(as_tuple=True)[1]
original_token_id = inputs["input_ids"][0, mask_token_index].clone()
inputs["input_ids"][0, mask_token_index] = tokenizer.mask_token_id
targets[0, mask_token_index] = original_token_id  # 해당 위치만 정답 레이블 설정

print(f"Original : {text}")
print(f"Masked   : {tokenizer.decode(inputs['input_ids'][0])}")
print(f"Targets  : {targets[0].tolist()}  (non-masked positions = -100)\n")

with torch.no_grad():
    outputs = model(**inputs, labels=targets)

loss = outputs.loss
logits = outputs.logits
mask_logits = logits[0, mask_token_index, :]
top_tokens = torch.topk(mask_logits, 10, dim=1)

print(f"Loss: {loss.item():.4f}\n")
print("Top predictions for [MASK]:")
for score, token_id in zip(top_tokens.values[0], top_tokens.indices[0]):
    token = tokenizer.decode([token_id.item()])
    print(f"  Token: '{token}', Score: {score.item():.4f}")